# Nutrition API clients — manual test

Exercises `calorie_tracker.nutrition_apis.usda` and `calorie_tracker.nutrition_apis.open_food_facts` directly, with real network calls against the live APIs. No AWS/DynamoDB involved — these two modules are pure HTTP clients with no dependency on the repository layer, so this notebook works with zero infrastructure set up.

Kernel: select **Python (calorie-tracker)**, which points at this project's `.venv`.

## USDA API key

`config.settings.usda_api_key` defaults to `DEMO_KEY` if unset, which is rate-limited to roughly 30 requests/hour/IP — easy to burn through across a few cells. If you have your own free key (https://api.data.gov/signup), copy `.env.example` to `.env` at the repo root and set `USDA_API_KEY=your-key-here` there — no code changes needed, and it'll be picked up automatically next time this kernel (re)starts, since `Settings()` reads `.env` once at import time.

In [ ]:
from calorie_tracker.config import settings

using_demo_key = settings.usda_api_key == "DEMO_KEY"
print("Using DEMO_KEY (rate-limited)" if using_demo_key else "Using a custom USDA_API_KEY from .env")


In [ ]:
from calorie_tracker.nutrition_apis import usda, open_food_facts
from calorie_tracker.models import Ingredient


def show(results: list[Ingredient]) -> None:
    if not results:
        print("(no results)")
        return
    for ing in results:
        m = ing.per_100g
        print(
            f"{ing.ingredient_id:22s} {ing.name[:42]:42s} src={ing.source:6s} "
            f"kcal={m.calories:6.1f} P={m.protein:5.1f} C={m.carbs:5.1f} F={m.fat:5.1f}  (per 100g)"
        )


## USDA FoodData Central

`dataType=Foundation,SR Legacy` filters to generic/whole foods (matches what a home-cooked meal needs), not branded products.

In [ ]:
show(usda.search("egg"))


In [ ]:
show(usda.search("chicken breast"))


In [ ]:
show(usda.search("white bread"))


In [ ]:
# A query unlikely to match anything with a calories value — should print "(no results)"
show(usda.search("asdkfjhasdkfjh"))


## Open Food Facts

Uses the Search-a-licious API (`search.openfoodfacts.org`) — the legacy `cgi/search.pl` endpoint is currently broken and isn't used. Better than USDA for packaged/branded products, and multilingual (relevant since meals will often be described in Polish).

In [ ]:
show(open_food_facts.search("cheddar cheese"))


In [ ]:
show(open_food_facts.search("kefir"))


In [ ]:
# Polish query — worth checking since real usage will often be in Polish
show(open_food_facts.search("jajko"))


## Notes

- Both `search()` functions skip results missing a real (non-null) calories value — see `Task 4`'s fix round in the plan for why that guard exists.
- This notebook does **not** exercise `services.ingredient_resolution.resolve()` (the cache-first orchestration layer that also calls these two clients) since that needs a real or mocked DynamoDB — see `tests/test_ingredient_resolution.py` for that, which mocks these two `search()` calls.